# Model Comparison: xLSTM Models

This notebook evaluates and compares three music generation models using various metrics:
1. **Musical Quality Metrics**: Pitch range, key stability, note density, etc.
2. **Self-Similarity Matrix (SSM) Metrics**: Structure analysis.
3. **Similarity Error (SE)**: Comparison against real validation data.

## Models
- **Model 1**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000`
- **Model 2**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749`
- **Model 3**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134`


In [1]:
import os
import pandas as pd
from pathlib import Path
import json

# Define Paths
BASE_DIR = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation")
SCRIPTS_DIR = BASE_DIR
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

MODEL_1_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000")
MODEL_2_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749")
MODEL_3_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134")

# Validation Data for SE
REAL_LIST = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt")
REAL_DIR = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/midi")

# Python Executable (muspy-env)
PYTHON_EXEC = "/home/e20365/miniconda3/envs/muspy-env/bin/python"

print(f"Model 1: {MODEL_1_PATH}")
print(f"Model 2: {MODEL_2_PATH}")
print(f"Model 3: {MODEL_3_PATH}")

Model 1: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000
Model 2: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749
Model 3: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134


## 1. Musical Quality Metrics

In [2]:
def run_musical_metrics(model_path, label):
    print(f"--- Running Musical Metrics for {label} ---")
    out_csv = RESULTS_DIR / f"musical_metrics_{label}.csv"
    summary_csv = RESULTS_DIR / f"musical_summary_{label}.csv"
    
    # 1. Compute Metrics
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/musical_quality_metrics.py --input_dir "{model_path}" --out_csv "{out_csv}"'
    print(f"Executing: {cmd}")
    ret = os.system(cmd)
    if ret != 0: print(f"Error running musical metrics for {label}")
        
    # 2. Aggregate
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/aggregate_metrics.py --input_csv "{out_csv}" --output_csv "{summary_csv}"'
    print(f"Executing: {cmd}\n")
    ret = os.system(cmd)
    if ret != 0: print(f"Error aggregating metrics for {label}")
        
run_musical_metrics(MODEL_1_PATH, "model1")
run_musical_metrics(MODEL_2_PATH, "model2")
run_musical_metrics(MODEL_3_PATH, "model3")

--- Running Musical Metrics for model1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/musical_quality_metrics.py --input_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000" --out_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv"


[OK] Wrote 80 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_summary_model1.csv"



[INFO] Loaded 80 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv
[INFO] Rows after filtering errors: 80

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 31.5875 ± 19.2398             
key_stability             | 0.5211 ± 0.2855               
key_change_rate           | 0.4797 ± 0.3058               
chord_diversity           | 0.4283 ± 0.2168               
pitch_range               | 38.0000 ± 19.5221             
stepwise_ratio            | 0.5308 ± 0.2282               
note_density_mean         | 31.7950 ± 18.3028             
note_density_std          | 11.2843 ± 8.3111              
duration_entropy          | 1.0289 ± 0.5807               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fy

[OK] Wrote 60 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model2.csv
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model2.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_summary_model2.csv"



[INFO] Loaded 60 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model2.csv
[INFO] Rows after filtering errors: 60

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 39.0000 ± 17.1622             
key_stability             | 0.3191 ± 0.1800               
key_change_rate           | 0.6600 ± 0.1850               
chord_diversity           | 0.6562 ± 0.1730               
pitch_range               | 61.0000 ± 13.5071             
stepwise_ratio            | 0.3780 ± 0.1585               
note_density_mean         | 32.3524 ± 12.1580             
note_density_std          | 14.0892 ± 5.4857              
duration_entropy          | 1.4158 ± 0.4846               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fy

[OK] Wrote 80 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model3.csv
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model3.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_summary_model3.csv"



[INFO] Loaded 80 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model3.csv
[INFO] Rows after filtering errors: 80

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 38.6625 ± 18.6927             
key_stability             | 0.6349 ± 0.3000               
key_change_rate           | 0.3838 ± 0.3435               
chord_diversity           | 0.2303 ± 0.1948               
pitch_range               | 28.2405 ± 13.4338             
stepwise_ratio            | 0.4938 ± 0.2949               
note_density_mean         | 32.6008 ± 19.2353             
note_density_std          | 6.9629 ± 6.3309               
duration_entropy          | 1.0429 ± 0.5584               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fy

## 2. Structure Metrics (SSM)

In [3]:
def run_ssm_metrics(model_path, label):
    print(f"--- Running SSM Metrics for {label} ---")
    ssm_out_dir = RESULTS_DIR / f"ssm_outputs_{label}"
    metrics_csv = RESULTS_DIR / f"ssm_metrics_{label}.csv"
    summary_csv = RESULTS_DIR / f"ssm_summary_{label}.csv"
    
    # 1. Compute SSMs
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/compute_bar_ssm.py --input_dir "{model_path}" --output_dir "{ssm_out_dir}" --feature chroma --binarize'
    print(f"Executing: {cmd}")
    ret = os.system(cmd)
    if ret != 0: print(f"Error computing SSMs for {label}")
        
    # 2. Compute Metrics from SSMs
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/compute_metrics_from_ssm.py --ssm_dir "{ssm_out_dir}" --out_csv "{metrics_csv}"'
    print(f"Executing: {cmd}")
    ret = os.system(cmd)
    if ret != 0: print(f"Error computing SSM metrics for {label}")
        
    # 3. Aggregate
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/aggregate_metrics.py --input_csv "{metrics_csv}" --output_csv "{summary_csv}"'
    print(f"Executing: {cmd}\n")
    ret = os.system(cmd)
    if ret != 0: print(f"Error aggregating SSM metrics for {label}")

run_ssm_metrics(MODEL_1_PATH, "model1")
run_ssm_metrics(MODEL_2_PATH, "model2")
run_ssm_metrics(MODEL_3_PATH, "model3")

--- Running SSM Metrics for model1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_bar_ssm.py --input_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000" --output_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_outputs_model1" --feature chroma --binarize


[INFO] Found 80 MIDI files under: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000
[INFO] Output dir: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_outputs_model1
[INFO] Settings: fs=50, feature=chroma, binarize=True
[OK] temp_0.6_bars_16_001.mid -> bars=29 | saved: temp_0.6_bars_16_001__barSSM_chroma_fs50.npy, temp_0.6_bars_16_001__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_002.mid -> bars=16 | saved: temp_0.6_bars_16_002__barSSM_chroma_fs50.npy, temp_0.6_bars_16_002__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_003.mid -> bars=20 | saved: temp_0.6_bars_16_003__barSSM_chroma_fs50.npy, temp_0.6_bars_16_003__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_004.mid -> bars=19 | saved: temp_0.6_bars_16_004__barSSM_chroma_fs50.npy, temp_0.6_bars_16_004__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_005.mid -> bars=17 | saved: temp_0.6_bars_1

[OK] temp_0.8_bars_64_001.mid -> bars=51 | saved: temp_0.8_bars_64_001__barSSM_chroma_fs50.npy, temp_0.8_bars_64_001__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_002.mid -> bars=65 | saved: temp_0.8_bars_64_002__barSSM_chroma_fs50.npy, temp_0.8_bars_64_002__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_003.mid -> bars=64 | saved: temp_0.8_bars_64_003__barSSM_chroma_fs50.npy, temp_0.8_bars_64_003__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_004.mid -> bars=36 | saved: temp_0.8_bars_64_004__barSSM_chroma_fs50.npy, temp_0.8_bars_64_004__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_005.mid -> bars=63 | saved: temp_0.8_bars_64_005__barSSM_chroma_fs50.npy, temp_0.8_bars_64_005__barSSM_chroma_fs50.png
[OK] temp_0.9_bars_16_001.mid -> bars=16 | saved: temp_0.9_bars_16_001__barSSM_chroma_fs50.npy, temp_0.9_bars_16_001__barSSM_chroma_fs50.png
[OK] temp_0.9_bars_16_002.mid -> bars=17 | saved: temp_0.9_bars_16_002__barSSM_chroma_fs50.npy, temp_0.9_bars_16_002__barSSM_chroma_fs50.png
[OK] temp_0.9

/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


...

Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_metrics_from_ssm.py --ssm_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_outputs_model3" --out_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model3.csv"


/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate point

/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


[OK] Wrote 79 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model3.csv
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model3.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_summary_model3.csv"



[INFO] Loaded 79 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model3.csv

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 39.1519 ± 18.2891             
avg_offdiag               | 0.7297 ± 0.2178               
rep_density               | 0.6265 ± 0.2989               
struct_entropy            | 6.1431 ± 1.1436               
block_coherence           | 0.3845 ± 0.2413               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_summary_model3.csv


## 3. Similarity Error (SE)

In [4]:
def run_se_metric(model_path, label):
    print(f"--- Running SE Metric for {label} ---")
    out_json = RESULTS_DIR / f"se_results_{label}.json"
    
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/compute_similarity_error.py --real_list "{REAL_LIST}" --real_dir "{REAL_DIR}" --gen_dir "{model_path}" --out "{out_json}" --limit 100'
    print(f"Executing: {cmd}\n")
    ret = os.system(cmd)
    if ret != 0: print(f"Error computing SE for {label}")

run_se_metric(MODEL_1_PATH, "model1")
run_se_metric(MODEL_2_PATH, "model2")
run_se_metric(MODEL_3_PATH, "model3")

--- Running SE Metric for model1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/midi" --gen_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000" --out "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model1.json" --limit 100



[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000
[INFO] Found 100 real, 80 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.047075
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model1.json
--- Running SE Metric for model2 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generat

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749
[INFO] Found 100 real, 60 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.087100
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model2.json
--- Running SE Metric for model3 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_prepro

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134
[INFO] Found 100 real, 80 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.142697
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model3.json


## 4. Aggregate Results and Compare

In [5]:
def load_summary(csv_path):
    if not csv_path.exists():
        return {}
    df = pd.read_csv(csv_path)
    return dict(zip(df['Metric'], df['Formatted']))

def load_se(json_path):
    if not json_path.exists():
        return "N/A"
    with open(json_path, 'r') as f:
        data = json.load(f)
    return f"{data.get('SE', 0):.4f}"

# Load Data
data_m1 = load_summary(RESULTS_DIR / "musical_summary_model1.csv")
data_m1.update(load_summary(RESULTS_DIR / "ssm_summary_model1.csv"))
data_m1['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model1.json")

data_m2 = load_summary(RESULTS_DIR / "musical_summary_model2.csv")
data_m2.update(load_summary(RESULTS_DIR / "ssm_summary_model2.csv"))
data_m2['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model2.json")


data_m3 = load_summary(RESULTS_DIR / "musical_summary_model3.csv")
data_m3.update(load_summary(RESULTS_DIR / "ssm_summary_model3.csv"))
data_m3['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model3.json")

# Construct Comparison Table
all_metrics = sorted(list(set(data_m1.keys()) | set(data_m2.keys()) | set(data_m3.keys())))

# Prioritize key metrics order
priority = ['Similarity Error (SE)', 'pitch_range', 'key_stability', 'chord_diversity', 
            'note_density_mean', 'duration_entropy', 'block_coherence', 'struct_entropy']
remaining = [m for m in all_metrics if m not in priority]
ordered_metrics = [m for m in priority if m in all_metrics] + remaining

comparison_data = []
for m in ordered_metrics:
    comparison_data.append({
        "Metric": m,
        "Model 1 (Checkpoint 84000)": data_m1.get(m, "N/A"),
        "Model 2 (Jan 28 Batch)": data_m2.get(m, "N/A"),
        "Model 3 (Feb 25 Batch)": data_m3.get(m, "N/A")
    })

comp_df = pd.DataFrame(comparison_data)

# Save and Display
comp_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("\n=== Model Comparison Table ===")
print(comp_df.to_markdown(index=False))
comp_df


=== Model Comparison Table ===
| Metric                | Model 1 (Checkpoint 84000)   | Model 2 (Jan 28 Batch)   | Model 3 (Feb 25 Batch)   |
|:----------------------|:-----------------------------|:-------------------------|:-------------------------|
| Similarity Error (SE) | 0.0471                       | 0.0871                   | 0.1427                   |
| pitch_range           | 38.0000 ± 19.5221            | 61.0000 ± 13.5071        | 28.2405 ± 13.4338        |
| key_stability         | 0.5211 ± 0.2855              | 0.3191 ± 0.1800          | 0.6349 ± 0.3000          |
| chord_diversity       | 0.4283 ± 0.2168              | 0.6562 ± 0.1730          | 0.2303 ± 0.1948          |
| note_density_mean     | 31.7950 ± 18.3028            | 32.3524 ± 12.1580        | 32.6008 ± 19.2353        |
| duration_entropy      | 1.0289 ± 0.5807              | 1.4158 ± 0.4846          | 1.0429 ± 0.5584          |
| block_coherence       | 0.3858 ± 0.1871              | 0.3744 ± 0.1001        

,Metric,Model 1 (Checkpoint 84000),Model 2 (Jan 28 Batch),Model 3 (Feb 25 Batch)
0,Similarity Error (SE),0.0471,0.0871,0.1427
1,pitch_range,38.0000 ± 19.5221,61.0000 ± 13.5071,28.2405 ± 13.4338
2,key_stability,0.5211 ± 0.2855,0.3191 ± 0.1800,0.6349 ± 0.3000
3,chord_diversity,0.4283 ± 0.2168,0.6562 ± 0.1730,0.2303 ± 0.1948
4,note_density_mean,31.7950 ± 18.3028,32.3524 ± 12.1580,32.6008 ± 19.2353
5,duration_entropy,1.0289 ± 0.5807,1.4158 ± 0.4846,1.0429 ± 0.5584
6,block_coherence,0.3858 ± 0.1871,0.3744 ± 0.1001,0.3845 ± 0.2413
7,struct_entropy,5.3976 ± 1.7005,6.0465 ± 1.2837,6.1431 ± 1.1436
8,avg_offdiag,0.6305 ± 0.2097,0.5293 ± 0.1157,0.7297 ± 0.2178
9,bars_N,31.5875 ± 19.2398,39.0000 ± 17.1622,39.1519 ± 18.2891
